In [206]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [ ]:
# =========================
# PARÂMETROS
# =========================
ticker = "^GSPC" # código do ativo no yahoo
start_date = "1998-01-01"
end_date = None
ma_window = 200 # Média movel para ser testada


In [208]:
df = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    progress=False,
    auto_adjust=True,
    back_adjust=True,
    multi_level_index=False,
    
)

df = df.dropna()
df["ma"] = df["Close"].rolling(ma_window).mean()
df.dropna(inplace=True)



In [209]:
close = df['Close']
open  = df['Open']

ma = df["ma"]
m_flag = (close >= ma).astype(int)

# --- sinal 2: retorno mensal comparado ao mês anterior (r_flag) ---
# calcula (Close - Open) do mês
monthly_diff = (close.resample('ME').last() / open.resample('ME').first()) - 1

# compara com mês anterior
r_flag_monthly = (monthly_diff >= monthly_diff.shift(1)).astype(int)
r_flag = r_flag_monthly.reindex(close.index, method='ffill')

df['m_flag'] = m_flag
df['r_flag'] = r_flag
df.dropna(inplace=True)
df.tail()

,Close,High,Low,Open,Volume,ma,m_flag,r_flag
Date,,,,,,,,
2025-12-08,6846.509766,6878.270020,6827.189941,6875.200195,4757130000,6199.503989,1,0.0
2025-12-09,6840.509766,6864.919922,6837.430176,6840.609863,4508050000,6203.790288,1,0.0
2025-12-10,6886.680176,6900.669922,6824.689941,6833.490234,5526570000,6208.447439,1,0.0
2025-12-11,6901.000000,6903.459961,6833.450195,6861.299805,5021060000,6213.172139,1,0.0
2025-12-12,6827.410156,6899.850098,6801.790039,6886.850098,3163284000,6218.001340,1,0.0


In [210]:
df["position"] = np.where(
    ((df["m_flag"] == 1) & (df["r_flag"] == 1)),
    1,
    0
)

df.head()
df["ret"] = df["Close"].pct_change()
df["strategy_ret"] = df["position"] * df["ret"].shift(-1)

df.dropna(inplace=True)
df["buy_hold"] = df["ret"].cumsum()
df["strategy"] = df["strategy_ret"].cumsum()



In [211]:
# =========================
# PLOT
# =========================
fig = make_subplots(
    rows=1,
    cols=1,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}]]
)

# Buy & Hold
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2)
    ),
    secondary_y=False
)

# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Moving Average Strategy",
        line=dict(width=2)
    ),
    secondary_y=False
)

# Position (0 / 1)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["position"],
        name="Position",
        line=dict(width=1, dash="dot"),
        opacity=0.7
    ),
    secondary_y=True
)

# Layout
fig.update_layout(
    title="SP500 | Dual Momentum",
    xaxis_title="Date",
    yaxis_title="Cumulative Return (%)",
    yaxis2_title="Position",
    legend=dict(x=0.01, y=0.99),
    template="plotly_white",
    height=600
)

# Ajuste do eixo secundário
fig.update_yaxes(range=[-0.05, 1.05], secondary_y=True)

fig.show()



In [212]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-1]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    df["strategy_ret"].mean() / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    df["ret"].mean() / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 233.19%
Strategy Return:   99.14%

Buy & Hold Vol: 19.32%
Strategy Vol:   7.68%

Buy & Hold Sharpe: 0.45
Strategy Sharpe:   0.48
